# 06 — Cross-Linguistic Analysis

Compare EO operator distributions across 30 typologically diverse languages.
Analyze by language family, era (ancient vs modern), region, and morphological type.

In [ ]:
# ═══ Load cross-linguistic data ═══
import json
from pyodide.http import pyfetch
from collections import Counter, defaultdict

HELIX = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']

crossling = None
try:
    resp = await pyfetch('../data/crossling.json')
    crossling = json.loads(await resp.string())
    langs = crossling.get('languages', {})
    print(f'Loaded cross-linguistic data: {len(langs)} languages')
except:
    print('No cross-linguistic data found. Download treebanks in 01_Corpus first.')

In [ ]:
# ═══ Distribution table ═══
if crossling and 'languages' in crossling:
    langs = crossling['languages']
    
    # Sort by era then name
    era_order = {'ancient': 0, 'medieval': 1, 'modern': 2}
    sorted_langs = sorted(langs.keys(), key=lambda x: (
        era_order.get(langs[x].get('era', 'modern'), 3), x
    ))
    
    print(f'{"Language":>25s} {"n":>5s}', end='')
    for op in HELIX:
        print(f' {op:>6s}', end='')
    print(f'  {"Family"}')
    print('-' * 120)
    
    for lang in sorted_langs:
        d = langs[lang]
        era = {'ancient': '†', 'medieval': '‡', 'modern': ' '}.get(d.get('era', ''), '?')
        pcts = d.get('op_pcts', {})
        print(f'{era}{lang:>24s} {d.get("n_classified", d.get("n_verbs", 0)):>5d}', end='')
        for op in HELIX:
            print(f' {pcts.get(op, 0):>5.1f}%', end='')
        print(f'  [{d.get("family", "")}]')

In [ ]:
# ═══ SUP analysis across languages ═══
if crossling and 'languages' in crossling:
    langs = crossling['languages']
    
    sup_data = [(lang, d.get('op_pcts', {}).get('SUP', 0), d.get('n_classified', 0))
                for lang, d in langs.items()]
    sup_data.sort(key=lambda x: -x[1])
    
    print('SUP (Superposition) Operator Across Languages')
    print('=' * 70)
    for lang, pct, n in sup_data:
        d = langs[lang]
        bar = '█' * int(pct * 3)
        era = d.get('era', '')
        family = d.get('family', '')
        print(f'  {lang:>25s} {pct:>5.1f}% {bar:30s} [{family}, {era}]')

In [ ]:
# ═══ Analysis by ERA ═══
if crossling and 'languages' in crossling:
    langs = crossling['languages']
    eras = defaultdict(list)
    for lang, d in langs.items():
        eras[d.get('era', 'unknown')].append(d)
    
    print('Operator Distributions by Era')
    print('=' * 70)
    for era in ['ancient', 'medieval', 'modern']:
        members = eras.get(era, [])
        if not members: continue
        avg = {op: sum(m.get('op_pcts', {}).get(op, 0) for m in members) / len(members) for op in HELIX}
        print(f'\n  {era.upper()} ({len(members)} languages):')
        print(f'    ', end='')
        for op in HELIX:
            print(f'{op}:{avg[op]:4.1f}%  ', end='')
        print()
        
        # Triad sums
        exist = sum(avg[op] for op in ['NUL','DES','INS'])
        struct = sum(avg[op] for op in ['SEG','CON','SYN'])
        interp = sum(avg[op] for op in ['ALT','SUP','REC'])
        print(f'    Existence: {exist:.1f}%  Structure: {struct:.1f}%  Interpretation: {interp:.1f}%')

In [ ]:
# ═══ Analysis by FAMILY ═══
if crossling and 'languages' in crossling:
    langs = crossling['languages']
    families = defaultdict(list)
    for lang, d in langs.items():
        families[d.get('family', 'unknown')].append(d)
    
    print('Operator Distributions by Language Family')
    print('=' * 70)
    for family in sorted(families.keys()):
        members = families[family]
        avg = {op: sum(m.get('op_pcts', {}).get(op, 0) for m in members) / len(members) for op in HELIX}
        print(f'\n  {family} ({len(members)} lang):', end='  ')
        for op in HELIX:
            print(f'{op}:{avg[op]:4.1f}%', end='  ')
        print()

In [ ]:
# ═══ Load and inspect individual language verbs ═══
LANG_TO_INSPECT = 'Ancient_Greek'  # Change this!

try:
    resp = await pyfetch(f'../data/crossling/{LANG_TO_INSPECT}/classified.json')
    lang_data = json.loads(await resp.string())
    cls = lang_data.get('classifications', [])
    
    print(f'{LANG_TO_INSPECT}: {len(cls)} classified verbs')
    print(f'Family: {lang_data.get("family", "")} | Era: {lang_data.get("era", "")}')
    
    dist = Counter(c.get('operator', '?') for c in cls)
    print(f'\nDistribution:')
    for op in HELIX:
        count = dist.get(op, 0)
        pct = count / len(cls) * 100 if cls else 0
        print(f'  {op:>5s} {count:>5d} ({pct:>5.1f}%)')
    
    print(f'\nSample verbs:')
    for c in cls[:20]:
        print(f'  {c.get("verb",""):>20s} -> {c.get("operator","")} ({c.get("confidence","")}) {c.get("gloss","")}')
except Exception as e:
    print(f'Could not load {LANG_TO_INSPECT}: {e}')